In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# 1. Lendo o dataset diretamente da URL
url = "https://raw.githubusercontent.com/j-convey/BankTextCategorizer/main/data/data1.csv"
df = pd.read_csv(url, encoding='latin1')

# 2. Limpeza Inicial
# Vamos focar em 'Description' (Entrada) e 'Category' (Saída/Label)
df = df[['Description', 'Category']].dropna()

# 3. Pré-processamento simples (letras minúsculas e remoção de caracteres especiais)
df['Description'] = df['Description'].str.lower().str.replace(r'[^a-zA-Z\s]', '', regex=True)

print(f"Dataset carregado com {len(df)} registros.")
print("\nExemplo de dados:")
print(df.head())

# Exibir categorias únicas
print("\nCategorias únicas:")
print(df['Category'].unique())

Dataset carregado com 3470 registros.

Exemplo de dados:
              Description Category
0                   arbys     Food
1             burger king     Food
2                carls jr     Food
3               chickfila     Food
4  chipotle mexican grill     Food

Categorias únicas:
['Food' 'Home' 'Electronics' 'Entertainment' 'Utilities' 'Personal_Care'
 'Vacation' 'Clothes' 'Auto' 'Kids' 'Subscriptions_Memberships' 'Medical'
 'Personal Care' 'Debt' 'Savings']


In [20]:
# 1. O mapeamento manual de categorias (mapeamento_duckbill) foi removido conforme sua solicitação.
#    O modelo agora será treinado diretamente nas categorias originais do dataset.

# 2. Não aplicamos mapeamento no DataFrame. Usamos a coluna 'Category' original.
#    O DataFrame 'df' já contém as colunas 'Description' e 'Category' do carregamento inicial.

# 3. Treinando o Modelo com as categorias originais do seu projeto
X_train, X_test, y_train, y_test = train_test_split(
    df['Description'], df['Category'], test_size=0.2, random_state=42
)

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_v3 = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf_v3.fit_transform(X_train)

# Usando um classificador que lida melhor com poucos dados por categoria
from sklearn.linear_model import LogisticRegression
# Aplicando class_weight='balanced' para lidar com o desbalanceamento de classes
modelo_final = LogisticRegression(max_iter=1000, class_weight='balanced')
modelo_final.fit(X_train_tfidf, y_train)

def prever_original_cat(texto):
    vetor = tfidf_v3.transform([texto.lower()])
    pred = modelo_final.predict(vetor)
    prob = modelo_final.predict_proba(vetor).max()
    return pred[0], prob

# --- NOVO TESTE DE ESTRESSE ---
# Os testes usarão a função de prever com as categorias originais.
testes_finais = [
    "Starbucks Coffee", "McDonalds Burger", "Netflix Subscription",
    "Uber Ride", "Shell Gas", "Books at Amazon", "Pharmacy Pill"
]

print(f"{'DESCRIÇÃO':<20} | {'PREDIÇÃO':<15} | {'CONFIANÇA'}")
for item in testes_finais:
    cat, conf = prever_original_cat(item)
    print(f"{item:<20} | {cat:<15} | {conf:.2%}")

DESCRIÇÃO            | PREDIÇÃO        | CONFIANÇA
Starbucks Coffee     | Home            | 13.44%
McDonalds Burger     | Food            | 73.25%
Netflix Subscription | Subscriptions_Memberships | 34.18%
Uber Ride            | Entertainment   | 19.20%
Shell Gas            | Utilities       | 21.93%
Books at Amazon      | Electronics     | 17.07%
Pharmacy Pill        | Entertainment   | 11.80%


In [22]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Criar um Pipeline
# Isso encadeia o vetorizador e o classificador, o que é útil para GridSearchCV
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('logreg', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# 2. Definir a grade de parâmetros para o GridSearchCV
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],  # Experimentar com diferentes n-grams
    'tfidf__max_features': [5000, 10000, None],      # Limite de features do TF-IDF
    'logreg__C': [0.1, 1, 10]                        # Força de regularização da Regressão Logística
}

# 3. Configurar e executar o GridSearchCV
# Usamos 'f1_weighted' como métrica principal para considerar o desbalanceamento de classes
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

# 4. Exibir os melhores parâmetros e a melhor pontuação
print("\nMelhores parâmetros encontrados:", grid_search.best_params_)
print("Melhor F1-score (ponderado) no treinamento:", grid_search.best_score_)

# 5. Avaliar o modelo com os melhores parâmetros no conjunto de teste
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

print("\nAcurácia do Modelo Otimizado:", accuracy_score(y_test, y_pred_tuned))
print("\nRelatório de Classificação do Modelo Otimizado:\n", classification_report(y_test, y_pred_tuned))

Fitting 5 folds for each of 27 candidates, totalling 135 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(



Melhores parâmetros encontrados: {'logreg__C': 10, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}
Melhor F1-score (ponderado) no treinamento: 0.864268829273674

Acurácia do Modelo Otimizado: 0.8631123919308358

Relatório de Classificação do Modelo Otimizado:
                            precision    recall  f1-score   support

                     Auto       0.94      0.89      0.92        37
                  Clothes       0.95      0.88      0.91        83
              Electronics       0.98      0.95      0.96       140
            Entertainment       0.56      0.89      0.68        84
                     Food       0.85      0.61      0.71        28
                     Home       0.91      0.80      0.85       138
                     Kids       1.00      0.86      0.93        22
            Personal_Care       0.94      0.94      0.94        78
Subscriptions_Memberships       0.70      0.47      0.56        15
                Utilities       0.92      0.81      0.8

In [23]:
# Lista de testes para o modelo otimizado
novos_testes = [
    "Starbucks Coffee",
    "McDonalds Big Mac",
    "Netflix Monthly Bill",
    "Uber Trip to Office",
    "Shell Gas Station",
    "Amazon Echo Dot",
    "Pharmacy Prescription",
    "Nike Running Shoes",
    "Disney Plus Subscription",
    "Apple iPhone Case"
]

def testar_modelo_otimizado(lista_descricoes):
    print(f"{'DESCRIÇÃO':<25} | {'PREDIÇÃO':<20} | {'CONFIANÇA'}")
    print("-" * 65)
    for desc in lista_descricoes:
        # O best_model do GridSearchCV já inclui o TfidfVectorizer e o Classificador
        predicao = best_model.predict([desc.lower()])[0]
        # Obter a probabilidade da classe predita
        probabilidades = best_model.predict_proba([desc.lower()])
        confianca = probabilidades.max()

        print(f"{desc:<25} | {predicao:<20} | {confianca:.2%}")

testar_modelo_otimizado(novos_testes)

DESCRIÇÃO                 | PREDIÇÃO             | CONFIANÇA
-----------------------------------------------------------------
Starbucks Coffee          | Home                 | 39.21%
McDonalds Big Mac         | Food                 | 47.52%
Netflix Monthly Bill      | Subscriptions_Memberships | 84.30%
Uber Trip to Office       | Auto                 | 23.51%
Shell Gas Station         | Auto                 | 33.22%
Amazon Echo Dot           | Electronics          | 82.59%
Pharmacy Prescription     | Entertainment        | 17.43%
Nike Running Shoes        | Clothes              | 93.47%
Disney Plus Subscription  | Subscriptions_Memberships | 82.62%
Apple iPhone Case         | Electronics          | 98.02%


In [24]:
import joblib
from google.colab import files

# 1. Salvar o modelo otimizado (que inclui o pipeline com TF-IDF + LogisticRegression)
model_filename = 'modelo_categorizador_bancario.pkl'
joblib.dump(best_model, model_filename)

print(f"Modelo salvo com sucesso como: {model_filename}")

# 2. Comando para baixar o arquivo diretamente no seu computador
files.download(model_filename)

Modelo salvo com sucesso como: modelo_categorizador_bancario.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Como usar o modelo exportado em outro script:

```python
import joblib

# Carregar o modelo
modelo_carregado = joblib.load('modelo_categorizador_bancario.pkl')

# Fazer uma nova previsão
texto = "Uber trip"
predicao = modelo_carregado.predict([texto.lower()])[0]
print(f"Categoria: {predicao}")
```